In [1]:
import sys
import subprocess
import time
import json
from pathlib import Path

# Third-party Library Imports
import requests

# Optional imports (will check availability later)
try:
    import redis
    REDIS_AVAILABLE: bool = True
except ImportError:
    REDIS_AVAILABLE: bool = False

try:
    import psycopg2
    PSYCOPG2_AVAILABLE: bool = True
except ImportError:
    PSYCOPG2_AVAILABLE: bool = False

print("Libraries imported successfully.")
print(f"   Redis library available: {REDIS_AVAILABLE}")
print(f"   PostgreSQL library available: {PSYCOPG2_AVAILABLE}")

if not REDIS_AVAILABLE or not PSYCOPG2_AVAILABLE:
    print("\nSome libraries missing. Run 'uv sync' to install them.")

Libraries imported successfully.
   Redis library available: True
   PostgreSQL library available: True


In [2]:
try:
    docker_version_result = subprocess.run(["docker", "--version"], capture_output=True, text=True, timeout=5)
    if docker_version_result.returncode == 0:
        version_output = docker_version_result.stdout.strip()
        print(f"Docker is installed: {version_output}")
    else:
        print("Docker is installed but not working properly")
        print("Try restarting Docker Desktop")
except FileNotFoundError:
    print("Docker is not installed")
    print("Please install Docker Desktop from https://docs.docker.com/get-docker/")
except Exception as error:
    print(f"✗ Error checking Docker: {error}")


Docker is installed: Docker version 28.1.1, build 4eba377


In [3]:
try:
    compose_version_result = subprocess.run(["docker", "compose", "version"], capture_output=True, text=True, timeout=5)
    if compose_version_result.returncode == 0:
        compose_version = compose_version_result.stdout.split()[3]
        print(f"Docker Compose is available: {compose_version}")
    else:
        print("Docker Compose is not working properly")
        print("Make sure Docker Desktop is running")
except FileNotFoundError:
    print("Docker Compose command not found")
    print("Docker Compose should come with Docker Desktop")
except Exception as error:
    print(f"✗ Error checking Docker Compose: {error}")

Docker Compose is available: v2.35.1-desktop.1


### Run docker up commands

In [4]:
def test_postgres_connection():
    """Test if PostgreSQL is accessible and responding"""
    import socket
    
    try:
        # Test if PostgreSQL port is accessible
        test_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        test_socket.settimeout(3)
        connection_result = test_socket.connect_ex(('localhost', 5432))
        test_socket.close()
        
        if connection_result == 0:
            print("✓ PostgreSQL is accepting connections on port 5432")
            
            # Try to connect with actual credentials
            try:                
                database_connection = psycopg2.connect(
                    host="localhost",
                    port=5432,
                    database="dms_meta", 
                    user="dms",
                    password="dms"
                )
                
                print("✓ Successfully connected to database 'dms_meta'")
                
                # Test simple query
                cursor = database_connection.cursor()
                cursor.execute("SELECT version();")
                postgres_version = cursor.fetchone()[0]
                print(f"✓ PostgreSQL version: {postgres_version.split()[0]} {postgres_version.split()[1]}")
                
                cursor.close()
                database_connection.close()
                return True
                
            except ImportError:
                print("⚠ psycopg2 not installed - basic connection test only")
                return True
            except Exception as db_error:
                print(f"✗ Database connection failed: {db_error}")
                return False
                
        else:
            print("✗ PostgreSQL port 5432 is not accessible")
            print("Make sure the postgres service is running")
            return False
            
    except Exception as connection_error:
        print(f"✗ Could not test PostgreSQL connection: {connection_error}")
        return False


postgres_is_working = test_postgres_connection()

if postgres_is_working:
    print("\nDatabase connection details:")
    print("• Host: localhost")
    print("• Port: 5432") 
    print("• Database: dms_meta")
    print("• Username: dms")
    print("• Password: dms")

✓ PostgreSQL is accepting connections on port 5432
✓ Successfully connected to database 'dms_meta'
✓ PostgreSQL version: PostgreSQL 15.15

Database connection details:
• Host: localhost
• Port: 5432
• Database: dms_meta
• Username: dms
• Password: dms


In [5]:
def test_redis_connection():
    """Test if Redis is accessible and responding"""
    import socket
    
    try:
        # Test if Redis port is accessible
        test_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        test_socket.settimeout(3)
        connection_result = test_socket.connect_ex(('localhost', 6379))
        test_socket.close()
        
        if connection_result == 0:
            print("Redis is accepting connections on port 6379")
            
            # Try to connect with redis client if available
            try:                
                redis_client = redis.Redis(host='localhost', port=6379, decode_responses=True)
                
                # Test ping
                redis_response = redis_client.ping()
                if redis_response:
                    print("✓ Successfully connected to Redis")
                    
                    # Test basic operations
                    redis_client.set('test_key', 'test_value')
                    retrieved_value = redis_client.get('test_key')
                    
                    if retrieved_value == 'test_value':
                        print("✓ Redis read/write operations working")
                        redis_client.delete('test_key')  # cleanup
                        return True
                    else:
                        print("✗ Redis read/write test failed")
                        return False
                else:
                    print("✗ Redis ping failed")
                    return False
                    
            except ImportError:
                print("⚠ redis-py not installed - basic connection test only")
                return True
            except Exception as redis_error:
                print(f"✗ Redis connection failed: {redis_error}")
                return False
                
        else:
            print("Redis port 6379 is not accessible")
            print("Make sure the redis service is running")
            return False
            
    except Exception as connection_error:
        print(f"Could not test Redis connection: {connection_error}")
        return False


redis_is_working = test_redis_connection()

if redis_is_working:
    print("\nRedis connection details:")
    print("• Host: localhost")
    print("• Port: 6379")
    print("• Used for: Background job queue and caching")

Redis is accepting connections on port 6379
✓ Successfully connected to Redis
✓ Redis read/write operations working

Redis connection details:
• Host: localhost
• Port: 6379
• Used for: Background job queue and caching


In [6]:
from dotenv import load_dotenv
load_dotenv() 

True

In [7]:
def test_azure_blob_connection():
    """Test Azure Blob Storage connectivity and basic operations"""
    import os
    from datetime import datetime

    try:
        from azure.storage.blob import BlobServiceClient
    except ImportError:
        print("✗ azure-storage-blob not installed")
        return False

    conn_str = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
    container_name = os.getenv("AZURE_BLOB_CONTAINER")

    if not conn_str or not container_name:
        print("✗ Missing AZURE_STORAGE_CONNECTION_STRING or AZURE_BLOB_CONTAINER")
        return False

    try:
        blob_service = BlobServiceClient.from_connection_string(conn_str)
        container_client = blob_service.get_container_client(container_name)

        # Check container access
        try:
            container_client.get_container_properties()
            print(f"✓ Connected to Azure Blob container: {container_name}")
        except Exception:
            print(f"✗ Cannot access container '{container_name}'")
            return False

        # Test write/read/delete
        blob_name = f"_healthcheck/test_{datetime.utcnow().isoformat()}.txt"
        blob_client = container_client.get_blob_client(blob_name)

        test_data = b"azure blob test"
        blob_client.upload_blob(test_data, overwrite=True)

        downloaded = blob_client.download_blob().readall()

        if downloaded == test_data:
            print("✓ Azure Blob read/write operations working")
            blob_client.delete_blob()
            return True
        else:
            print("✗ Azure Blob read/write mismatch")
            return False

    except Exception as e:
        print(f"✗ Azure Blob test failed: {e}")
        return False

blob_ok = test_azure_blob_connection()

✓ Connected to Azure Blob container: credit-ocr


C:\Users\deril\AppData\Local\Temp\ipykernel_4196\2659228026.py:32: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  blob_name = f"_healthcheck/test_{datetime.utcnow().isoformat()}.txt"


✓ Azure Blob read/write operations working


In [8]:
def test_azure_openai_connection():
    """Test Azure OpenAI connectivity and inference"""
    import os

    try:
        from openai import AzureOpenAI
    except ImportError:
        print("✗ openai package not installed")
        return False

    endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
    api_key = os.getenv("AZURE_OPENAI_API_KEY")
    api_version = os.getenv("AZURE_OPENAI_API_VERSION")
    deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT")

    missing = [k for k, v in {
        "AZURE_OPENAI_ENDPOINT": endpoint,
        "AZURE_OPENAI_API_KEY": api_key,
        "AZURE_OPENAI_API_VERSION": api_version,
        "AZURE_OPENAI_DEPLOYMENT": deployment,
    }.items() if not v]

    if missing:
        print(f"✗ Missing Azure OpenAI env vars: {', '.join(missing)}")
        return False

    try:
        client = AzureOpenAI(
            api_key=api_key,
            azure_endpoint=endpoint,
            api_version=api_version,
        )

        response = client.chat.completions.create(
            model=deployment,
            messages=[
                {"role": "system", "content": "You are a health check."},
                {"role": "user", "content": "Reply with the word OK only."},
            ],
            temperature=0,
        )

        content = response.choices[0].message.content.strip()

        if content == "OK":
            print("✓ Azure OpenAI inference successful")
            return True
        else:
            print(f"✗ Unexpected Azure OpenAI response: {content}")
            return False

    except Exception as e:
        print(f"✗ Azure OpenAI test failed: {e}")
        return False

openai_ok = test_azure_openai_connection()

✓ Azure OpenAI inference successful
